# Variational AutoEncoder For Limb Generation
So heres the idea - the 7 measurements that we have don't fully define the geometry of these limbs. Not even close. Shape and bend and whatnot don't show up in measurement so what we're going to do is build an auto encoder for reconstructing the limb from its coefficients. We condition the decoder section on an embedding (Nonsense numbers) and the measurements instead of just reconstructing from the measurements themselves. That way, we have a few values we can play around with that should generate limbs with the same measurements but varying around the final shape of the limb. Insha'Allah. 

In [74]:
# | export_section
import torch
from torch import nn
import igl
from SSM_Driver import (
    Measurements,
    get_measurement_details,
    LegMeasurementDataset,
    create_examples,
    MeasurementLoss,
    RelativeMeasurementLoss,
    PointwiseLoss,
)
# | end_section

from tqdm import tqdm
import wandb

In [75]:
config = {
    "lr": 1e-2,
    "eta_min": 1e-5,
    "batch_size": 256,
    "log": False,
    "seed": 42,
    "epochs":500,
    "kl_coeff": 0.005,
    "mse_coeff":0.01,
    "scale": False,
    "normalise": True,
    "relativise": True,
}

torch.manual_seed(config["seed"])

if config["log"]:
    project="Open Limb SSM"
    name="VAE Extended Depth V0.1"
    if name in [x.name for x in wandb.Api().runs(project)]:
        raise ValueError("Duplicate name")
    wandb.init(project=project, name=name, config=config)

In [76]:
dtype = torch.float32
mean_verts = torch.load("./data_components/mean_verts.pt").to(dtype)
face2vert = torch.load("./data_components/face2vert.pt")

edge2vert, face2edge, edge2face = igl.edge_topology(
    mean_verts.numpy(), face2vert.numpy()
)

edge2vert = torch.from_numpy(edge2vert)
face2edge = torch.from_numpy(face2edge)
edge2face = torch.from_numpy(edge2face)

measurement_details = get_measurement_details(dtype)
measure = Measurements(
    edge2vert,
    face2edge,
    measurement_details,
)

In [77]:
class Encoder(nn.Module):
    def __init__(self, input, output):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
        )
        self.mu = nn.Linear(128, output)
        self.logvar = nn.Linear(128, output)

    def forward(self, x):
        x = self.net(x)
        return self.mu(x), self.logvar(x)

class Decoder(nn.Module):
    def __init__(self, input, output):
        super().__init__()
        # Input: 3 (z) + 7 (measurements) = 10
        self.net = nn.Sequential(
            nn.Linear(input, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, output) 
        )

    def forward(self, x):
        return self.net(x)
    
class VAE(nn.Module):
    def __init__(self, encoder_input, z_dim, measurement_dim, decoder_output, output_transform):
        super().__init__()
        self.encoder = Encoder(encoder_input, z_dim)
        self.decoder = Decoder(z_dim + measurement_dim, decoder_output)
        self.z_dim = z_dim
        self.output_transform = output_transform

    def forward(self, measurements, coefficients):
        mean, log_var = self.encoder(torch.cat((measurements, coefficients), 1))
        
        # Reparameterisation?
        epsilon = torch.randn_like(mean, dtype=coefficients.dtype).to(measurements.device)
        z = mean + torch.exp(0.5*log_var)*epsilon

        x_hat = self.decoder(torch.cat((measurements, z), 1))

        return x_hat* self.output_transform[1:] + self.output_transform[:1], mean, log_var
    
    def sample(self, measurements, samples=64):
        raise NotImplementedError
        self.eval()
        with torch.no_grad():
            z = torch.randn(samples, self.z_dim).to(measurements.device)

            # This is definitely not right but Im trying to figure out what I want from it. 
            # Do I want to have (measurements, samples, coeffs) or (samples, coeffs) from a single measurement?
            decoder_input = torch.cat((z, measurements), dim=-1)

            coeffs = self.decoder(decoder_input)

            




In [78]:
class VAEKLLoss(nn.Module):
    def forward(self, mean, log_var):
        """
        KL divergence between q(z|x) = N(mean, var) and p(z) = N(0, I)
        """
        return -0.5 * torch.mean(torch.sum(1 + log_var - mean.pow(2) - log_var.exp(), dim=1))

In [79]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scale = False
normalise = False
relativise = False

dataset = LegMeasurementDataset(
    measure,
    batch_size=config["batch_size"],
    scale=config["scale"],
    normalise=config["normalise"],
    relativise=config["relativise"],
    dtype=dtype,
    device="cpu",
)
dataloader = torch.utils.data.DataLoader(
    dataset, config["batch_size"], shuffle=False, num_workers=0
)
# loss_func = MeasurementLoss(dataset).to(device)
# loss_func = RelativeMeasurementLoss(dataset).to(device)
loss_func = PointwiseLoss(dataset).to(device)
kl_div = VAEKLLoss().to(device)
component_loss = nn.MSELoss().to(device)

# Add one for the scale factor
component_transforms = (
    torch.load("./data_components/scaled_component_transforms.pt")[:, : 10 + scale]
    .to(dtype)
    .to(device)
)
model = VAE(
    dataset.raw_components.shape[0] + config["scale"] + len(measurement_details),
    3,
    len(measurement_details),
    dataset.raw_components.shape[0] + config["scale"],
    component_transforms,
)
print(
    f"VAE input dims: \n\tInput: {dataset.raw_components.shape[0] + config['scale'] + len(measurement_details)}\n\tNonsense: {3}\n\tMeasurements: {len(measurement_details)}\n\tOutputs: {dataset.raw_components.shape[0] + config["scale"]}"
)

optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, config["epochs"])

VAE input dims: 
	Input: 17
	Nonsense: 3
	Measurements: 7
	Outputs: 10


In [ ]:
from SSM_Driver import LegMeasurementDataset as LMD
from SSM_Driver import measure
data = LMD(measure)

In [80]:
i = 0
best_loss = torch.inf
pbar = tqdm(total=config["epochs"], desc="Training")
for measures, components in dataloader:
    measures = measures.to(device)
    components = components.to(device)
    
    preds, means, log_vars = model(measures, components)

    kl_loss = kl_div(means, log_vars)
    mse_loss = component_loss(preds, components)
    main_loss = loss_func(preds, components)
    # loss = main_loss + config["kl_coeff"]*kl_loss + config["mse_coeff"]*mse_loss
    loss = config["kl_coeff"]*kl_loss + config["mse_coeff"]*mse_loss
    
    loss.backward()
    # # Gradient Clipping
    # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    optimizer.zero_grad()

    with torch.no_grad():
        pred_measures = dataset.get_measures(preds.to("cpu"), None).to(device)
        measures = dataset.get_measures(components.to("cpu"), None).to(device)

        # Approximate scale to get rough metric values
        if not config["scale"]:
            pred_measures *= 391
            measures *= 391

        # Calculate per measurement statistics
        mean_diff_per_measure = torch.mean((pred_measures - measures).abs() / measures, dim=0)
        max_diff_per_measure = torch.max((pred_measures - measures).abs() / measures, dim=0)[0]
        raw_diff_per_measure = torch.mean((pred_measures - measures).abs(), dim=0)
        raw_diff_per_measure = torch.max((pred_measures - measures).abs(), dim=0)[0]
        per_measure_details = {}
        for idx, detail in enumerate(measurement_details):
            per_measure_details |= {
                f"mean_diff_{detail['name']}": mean_diff_per_measure[idx].item(),
                f"max_diff_{detail['name']}": max_diff_per_measure[idx].item(),
                f"raw_diff_{detail['name']}": raw_diff_per_measure[idx].item(),
                f"raw_max_diff_{detail['name']}": raw_diff_per_measure[idx].item(),
            }

        if torch.isnan(loss).item():
            print(f"means stats: min={means.min()}, max={means.max()}, mean={means.mean()}")
            print(f"log_vars stats: min={log_vars.min()}, max={log_vars.max()}, mean={log_vars.mean()}")
            print(f"preds stats: min={preds.min()}, max={preds.max()}, mean={preds.mean()}")
            print({"loss": loss,
                "main_loss": main_loss,
                "component loss":  mse_loss.item(),
                "kl div loss":  kl_loss.item()})
            break

        if config["log"]:
            wandb.log({
                "loss": loss,
                "main_loss": main_loss,
                "component loss":  mse_loss.item(),
                "kl div loss":  kl_loss.item(),
                "measurement difference": torch.mean((pred_measures - measures).abs() / measures).item(),
                "maximum measurement difference": torch.max((pred_measures - measures).abs() / measures).item(),
                "maximum measurement difference raw": torch.max((pred_measures - measures).abs()).item(),
                "minimum difference": torch.min((pred_measures - measures).abs() / measures).item(),
                "scale":torch.mean(preds[:,-1]).item(),
                "scale min":torch.min(preds[:,-1]).item(),
                "scale max":torch.max(preds[:,-1]).item(),
                "scale std":torch.std(preds[:,-1]).item(),
            } | per_measure_details)
        # else:
        #     print({"loss": loss,
        #         "main_loss": main_loss,
        #         "component loss":  mse_loss.item(),
        #         "kl div loss":  kl_loss.item()})
            
        if loss < best_loss:
            torch.save(model.state_dict(), "models/bestVAE.pt")
            best_loss = loss

        if i and i % 5 == 0:
            create_examples(components.to("cpu"), preds.to("cpu"), dataset)

    if i >= config["epochs"]:
        break
    
    scheduler.step()
    pbar.update(1)
    pbar.set_postfix(loss=f"{loss.item():.4f}    {mse_loss.item():.4f}")
    i += 1

pbar.close()

Training:  78%|███████▊  | 391/500 [1:27:12<24:18, 13.38s/it, loss=0.0903    7.5597]                             
